# Multi-Query Attention (MQA) — Exercise

Companion to [Attention Part 2 § Multi-Query Attention](https://tanulsingh.github.io/blog/attention-architectures#multi-query-attention-shazeer-2019).

In standard MHA, every query head has its own K and V. In **MQA**, all `n_heads` query heads share a **single** K head and a **single** V head:

$$q_i = x W_Q^{(i)}, \quad k = x W_K, \quad v = x W_V$$

Result: KV cache shrinks by `n_heads`× (e.g., 32× smaller for a 32-head model).

You'll modify the `MultiHeadAttention` class from Part 1 into a `MultiQueryAttention`.

## Setup

In [ ]:
# TODO: imports — torch, torch.nn as nn, math

## TODO 1 — `MultiQueryAttention`

Starting from `MultiHeadAttention`, change two things:

1. **W_k** projects to `d_head` (not `d_model`) — produces 1 K head, not `n_heads`
2. **W_v** projects to `d_head` (not `d_model`) — produces 1 V head, not `n_heads`

W_q and W_o are unchanged. In `forward`:
- Q is reshaped/transposed as before → `(batch, n_heads, seq, d_head)`
- K, V are simpler: `(batch, seq, d_head)` → `(batch, 1, seq, d_head)` so they broadcast across all heads when computing `Q @ K.transpose(-2, -1)`.

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, causal: bool = False):
        super().__init__()
        # TODO 1a: store d_model, n_heads, d_head, causal
        # TODO 1b: W_q: nn.Linear(d_model, d_model)            # n_heads * d_head
        # TODO 1c: W_k: nn.Linear(d_model, d_head)             # ONE head only
        # TODO 1d: W_v: nn.Linear(d_model, d_head)             # ONE head only
        # TODO 1e: W_o: nn.Linear(d_model, d_model)
        pass

    def forward(self, x):
        # x: (batch, seq, d_model)
        # TODO 1f: project to Q, K, V
        # TODO 1g: reshape Q to (batch, n_heads, seq, d_head)
        # TODO 1h: reshape K, V to (batch, 1, seq, d_head)  # broadcast over heads
        # TODO 1i: compute attention with optional causal mask
        # TODO 1j: reshape back and apply W_o
        pass

In [ ]:
# Sanity check:
#   - mqa = MultiQueryAttention(d_model=64, n_heads=8)
#   - out = mqa(torch.randn(2, 10, 64))
#   - out.shape == (2, 10, 64)
#   - Compare param count: should be smaller than MultiHeadAttention(64, 8)
#     because W_k and W_v are now d_model × d_head instead of d_model × d_model

## TODO 2 — Verify the KV cache savings

For a forward on input `(batch, seq, d_model)`, the cache size is:
- MHA: `2 × n_heads × seq × d_head` values
- MQA: `2 × 1 × seq × d_head` values

Verify: count parameters in W_k of your MQA module — should be `d_model * d_head`, not `d_model * d_model`.

In [ ]:
# TODO 2:
#   - For d_model=64, n_heads=8 → d_head=8
#   - W_k.weight should be shape (8, 64) — that's d_head=8 output features × d_model=64 input
#   - vs MHA's W_k.weight which would be (64, 64)
#   - Print the param counts to compare

## Run the tests

In [ ]:
from tests import run_mqa_tests
# run_mqa_tests(MultiQueryAttention)